In [ ]:
from ada_fsaudit_bridge import load_dataset, lower_bound, upper_bound

salaries = load_dataset('salaries')
N = len(salaries)

def lower(k, n, alpha, popn=None, dist=None):
    if dist is None:
        dist = 'hyper'
    if popn is None and dist == 'binom':
        popn = n
    return lower_bound(k=k, n=n, alpha=alpha, popn=popn, dist=dist)

def upper(k, n, alpha, popn=None, dist=None):
    if dist is None:
        dist = 'hyper'
    if popn is None and dist == 'binom':
        popn = n
    return upper_bound(k=k, n=n, alpha=alpha, popn=popn, dist=dist)


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2, norm

def head(obj, n=6):
    if hasattr(obj, 'head'):
        return obj.head(n)
    return obj[:n]

sns.set_theme(style='whitegrid')


In [ ]:
import numpy as np
from math import sqrt
from scipy.stats import binom, chi2, chisquare, f, hypergeom, nct, norm, poisson, t


## Exercise 2.1 Drawing a sample


In this Chapter we will use one single data file, so you can get familiar with estimating a mean, a proportion, and their respective confidence intervals without learning the specifics of many different datasets. The `salaries` dataset contains salary details of 2,222 employees of the imaginary town of Winesbury, and is included with the `FSaudit` package. The data file indicates for each employee the Grade, Step within the Grade, and Gross salary, as well as a unique employee ID. The field Gender is coded with a 1 for female and a 0 for male.
In order to replicate the experiment in the book, you need to use the same random number seed as we did. Note that as of version 3.6, fundamental changes have been made in the random number generator. The default setting of the random number generator should normally not be changed, but for the benefit of readers who use an older version of `R` we set it to `Rounding`, which was the default prior to version 3.6.
Details of the data file can be obtained using base `R` functionality.


In [ ]:
display(list(salaries.columns))
N = len(salaries)
print(N)


We sample 50 units from the `salaries` dataset by using the `sample()` function.


In [ ]:
# RNGkind() is R-specific; NumPy RNG is used in Python
n = 50
np.random.seed(12345)
sample1 = salaries.iloc[np.random.choice(N, size=n, replace=False), :]
sample1.head()


## Exercise 2.2 Point estimation of the mean


The point estimate of the mean ($\bar{y}$) is obtained by dividing the sum of $y$ in the sample by the sample size. In `R` we simply use the `mean` function.


In [ ]:
y_bar = np.mean(sample1["gross"])


We can use the sample mean as an estimate for the total payroll in the population. The estimate of the total monthly payroll is then obtained by multiplying the sample mean with the number of elements in the population $N$.


In [ ]:
N * y_bar


## Exercise 2.3 Confidence intervals around the mean


First, we estimate the standard deviation in the population from the standard deviation of the sample.


In [ ]:
s = np.std(sample1["gross"])


The calculation of the standard error is then


In [ ]:
se = s / sqrt(n)


For a 95% confidence interval the t-value is based on tail probabilities of 0.025 and 0.095.


In [ ]:
tval = t.ppf(0.975, df=(n - 1))


We multiply the $t$-value with the standard error to obtain the precision achieved.


In [ ]:
precAch = tval * se


The resulting lower and upper bounds are


In [ ]:
lowerBound = y_bar - precAch
upperBound = y_bar + precAch


For a 99\% confidence interval, the calculations are


In [ ]:
from IPython.display import display
lowerBound = y_bar + t.ppf(0.005, df=49) * se
upperBound = y_bar + t.ppf(0.995, df=49) * se
display(N * lowerBound)
display(N * upperBound)


## Exercise 2.4 Extending the sample


To achieve the target precision of $E = $ 600,000, we extend the sample from Exercise 3.1 from 50 to 131 sampling units.


In [ ]:
np.random.seed(12345)
sample2 = salaries.iloc[np.random.choice(N, size=131, replace=False), :]


The mean of the total sample is:


In [ ]:
np.mean(sample2["gross"])


## Exercise 2.5 Finite populations


We saw in Section 3.2.3 that the finiteness of the population from which we sample has a noticeable effect when the sample size is greater than 10\% of the population size. In the example of the mean monthly payroll ($n = 50$, $N = 2,222$), it can be expected that this is the case if we want the precision around the total monthly payroll to be less than 300,000.
The confidence interval incorporating the finite-population correction factor is given by Equation 3.10:


In [ ]:
E = 300000
tval = t.ppf(0.975, df=(n - 1))
gamma = E**2 / (N * tval**2 * np.std(sample1["gross"])**2)
(N / (1 + gamma))


The total sample size required to obtain an estimate of the mean monthly payroll with a precision of 300,000 at 95% confidence is 424. This is more than 10% of the population size; therefore, we were right to use the finite-population correction factor. If we had not anticipated that the required sample size would have been unnecessarily large, not taking the finite-population correction factor into account would have resulted in a sample size of 524.


In [ ]:
(N**2 * tval**2) * np.std(sample1["gross"])**2 / E**2


## Exercise 2.6 Confidence intervals around the proportion


These bounds are calculated in R with the `upper` and `lower` functions, that are part of the `FSaudit` package.


In [ ]:
lower(k = 31, popn = 2222, n = 50, alpha = .025)
upper(k = 31, popn = 2222, n = 50, alpha = .025)


Binomial bounds can be obtained, by adding the distribution to the respective functions.


In [ ]:
lower(k = 31, n = 50, alpha = .025, dist = "binom")
upper(k = 31, n = 50, alpha = .025, dist = "binom")
lower(k = 31, popn = 2222, n = 50, alpha = .025, dist = "binom")
upper(k = 31, popn = 2222, n = 50, alpha = .025, dist = "binom")


Note that the hypergeometric distribution provides a more precise result.
